# PV project

## Loading the selected data into database

In [ ]:
## Install the required libraries
!pip install duckdb pandas numpy scipy streamlit langchain-community

## Import Libraries

In [ ]:
import duckdb
import os
from pathlib import Path

## Identify the Folder Locations

In [14]:
import os
print("Current Working Directory:", os.getcwd())
print("Contents of CWD:", os.listdir("."))


Current Working Directory: /Users/shymashameer/Desktop/Final Project/faers/notebooks
Contents of CWD: ['.DS_Store', 'PV Project.ipynb', '.ipynb_checkpoints']


In [ ]:
# Check current working directory
cwd = Path.cwd()
print("Current Working Directory:", cwd)

# Walk up to 'faers' project root
project_root = cwd.parent if cwd.name == "notebooks" else cwd
print("Project Root:", project_root)

# List all directories and files under project_root
print("\n--- Project Tree ---")
for root, dirs, files in os.walk(project_root):
    # Skip hidden folders like .git, .ipynb_checkpoints
    dirs[:] = [d for d in dirs if not d.startswith('.')]
    
    level = root.replace(str(project_root), '').count(os.sep)
    indent = ' ' * 4 * (level)
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 4 * (level + 1)
    for f in files:
        if not f.startswith('.'):
            print(f"{subindent}{f}")

Current Working Directory: /Users/shymashameer/Desktop/Final Project/faers/notebooks
Project Root: /Users/shymashameer/Desktop/Final Project/faers

--- Project Tree ---
faers/
    data/
        faers_db.duckdb
        faers_db.duckdb.wal
        raw/
            faers_ascii_2015q1/
                FAQs.pdf
                Readme.pdf
                ascii/
                    drug15q1.pdf
                    REAC15Q1.txt
                    OUTC15Q1.txt
                    reac15q1.pdf
                    DRUG15Q1.txt
                    ASC_NTS.pdf
                    outc15q1.pdf
                    INDI15Q1.txt
                    RPSR15Q1.txt
                    ther15q1.pdf
                    DEMO15Q1.txt
                    rpsr15q1.pdf
                    THER15Q1.txt
                    indi15q1.pdf
                    demo15q1.pdf
            faers_ascii_2017q2/
                FAQs.pdf
                Readme.pdf
                ascii/
                    drug17q2.pdf
        

## Set the project paths

In [ ]:
# Resolve project paths

# Set project root to parent directory of 'notebooks'
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR

# Define absolute paths relative to project root
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
DB_PATH = PROJECT_ROOT / "data" / "faers_db.duckdb"

## Exploring the files in the folder

In [27]:

print("--- Searching for ALL files under project root ---")
found_files = []
for root, dirs, files in os.walk(project_root):
    for f in files:
        if not f.startswith('.'):  # Ignore hidden files
            full_path = os.path.join(root, f)
            found_files.append(full_path)

print(f"Total files found in project: {len(found_files)}\n")

# Filter and display all .txt or .pdf files found
data_files = [f for f in found_files if f.lower().endswith(('.txt', '.pdf'))]
print("--- Data / PDF Files Found ---")
for df in data_files[:20]:  # Show first 20
    print(df)

--- Searching for ALL files under project root ---
Total files found in project: 417

--- Data / PDF Files Found ---
/Users/shymashameer/Desktop/Final Project/faers/data/raw/faers_ascii_2015q1/FAQs.pdf
/Users/shymashameer/Desktop/Final Project/faers/data/raw/faers_ascii_2015q1/Readme.pdf
/Users/shymashameer/Desktop/Final Project/faers/data/raw/faers_ascii_2015q1/ascii/drug15q1.pdf
/Users/shymashameer/Desktop/Final Project/faers/data/raw/faers_ascii_2015q1/ascii/REAC15Q1.txt
/Users/shymashameer/Desktop/Final Project/faers/data/raw/faers_ascii_2015q1/ascii/OUTC15Q1.txt
/Users/shymashameer/Desktop/Final Project/faers/data/raw/faers_ascii_2015q1/ascii/reac15q1.pdf
/Users/shymashameer/Desktop/Final Project/faers/data/raw/faers_ascii_2015q1/ascii/DRUG15Q1.txt
/Users/shymashameer/Desktop/Final Project/faers/data/raw/faers_ascii_2015q1/ascii/ASC_NTS.pdf
/Users/shymashameer/Desktop/Final Project/faers/data/raw/faers_ascii_2015q1/ascii/outc15q1.pdf
/Users/shymashameer/Desktop/Final Project/faers

In [29]:
## Searching for .txt files

print(f"Searching in: {RAW_DATA_DIR}\n")

# Find all files ending in .txt / .TXT (case-insensitive)
all_txt_files = []
for root, dirs, files in os.walk(RAW_DATA_DIR):
    for f in files:
        if f.lower().endswith('.txt'):
            all_txt_files.append(os.path.join(root, f).replace("\\", "/"))

print(f"Total .txt files discovered: {len(all_txt_files)}")

# Group files by type

demo_files = []
drug_files = []
reac_files = []

# Scan every file without restricting extension conditions
for root, _, files in os.walk(PROJECT_ROOT):
    for f in files:
        if f.startswith('.'):
            continue
        
        full_path = os.path.join(root, f).replace("\\", "/")
        f_upper = f.upper()
        
        # Match filenames containing the table names
        if "DEMO" in f_upper and (f_upper.endswith(".TXT") or "." not in f):
            demo_files.append(full_path)
        elif "DRUG" in f_upper and (f_upper.endswith(".TXT") or "." not in f):
            drug_files.append(full_path)
        elif "REAC" in f_upper and (f_upper.endswith(".TXT") or "." not in f):
            reac_files.append(full_path)

print(f"Total DEMO files found: {len(demo_files)}")
print(f"Total DRUG files found: {len(drug_files)}")
print(f"Total REAC files found: {len(reac_files)}\n")

if demo_files:
    print(f"Sample DEMO path: {demo_files[-1]}")
if drug_files:
    print(f"Sample DRUG path: {drug_files[-1]}")
if reac_files:
    print(f"Sample REAC path: {reac_files[-1]}\n")

# Connect to DuckDB and inspect schemas
con = duckdb.connect()

if demo_files:
    print("=" * 60)
    print("📋 DEMO FILE SCHEMA & COLUMNS")
    print("=" * 60)
    display(con.execute(f"DESCRIBE SELECT * FROM read_csv_auto('{demo_files[0]}', delim='$', header=True, ignore_errors=True);").df()[['column_name', 'column_type']])

if drug_files:
    print("\n" + "=" * 60)
    print("📋 DRUG FILE SCHEMA & COLUMNS")
    print("=" * 60)
    display(con.execute(f"DESCRIBE SELECT * FROM read_csv_auto('{drug_files[0]}', delim='$', header=True, ignore_errors=True);").df()[['column_name', 'column_type']])

if reac_files:
    print("\n" + "=" * 60)
    print("📋 REAC FILE SCHEMA & COLUMNS")
    print("=" * 60)
    display(con.execute(f"DESCRIBE SELECT * FROM read_csv_auto('{reac_files[0]}', delim='$', header=True, ignore_errors=True);").df()[['column_name', 'column_type']])

con.close()

Searching in: /Users/shymashameer/Desktop/Final Project/faers/data/raw

Total .txt files discovered: 173
Total DEMO files found: 24
Total DRUG files found: 24
Total REAC files found: 24

Sample DEMO path: /Users/shymashameer/Desktop/Final Project/faers/data/raw/faers_ascii_2018q2/ascii/DEMO18Q2.txt
Sample DRUG path: /Users/shymashameer/Desktop/Final Project/faers/data/raw/faers_ascii_2018q2/ascii/DRUG18Q2.txt
Sample REAC path: /Users/shymashameer/Desktop/Final Project/faers/data/raw/faers_ascii_2018q2/ascii/REAC18Q2.txt

📋 DEMO FILE SCHEMA & COLUMNS


,column_name,column_type
0,primaryid,BIGINT
1,caseid,BIGINT
2,caseversion,BIGINT
3,i_f_code,BOOLEAN
4,event_dt,BIGINT
5,mfr_dt,BIGINT
6,init_fda_dt,BIGINT
7,fda_dt,BIGINT
8,rept_cod,VARCHAR
9,auth_num,VARCHAR



📋 DRUG FILE SCHEMA & COLUMNS


,column_name,column_type
0,primaryid,BIGINT
1,caseid,BIGINT
2,drug_seq,BIGINT
3,role_cod,VARCHAR
4,drugname,VARCHAR
5,prod_ai,VARCHAR
6,val_vbm,BIGINT
7,route,VARCHAR
8,dose_vbm,VARCHAR
9,cum_dose_chr,DOUBLE



📋 REAC FILE SCHEMA & COLUMNS


,column_name,column_type
0,primaryid,BIGINT
1,caseid,BIGINT
2,pt,VARCHAR
3,drug_rec_act,VARCHAR


## SGLT2 Inhibitors

In [24]:
## SGLT2 Inhibitors
def collect_faers_files(prefix):
    matched_files = []
    for root, _, files in os.walk(RAW_DATA_DIR):
        for f in files:
            if f.startswith('.'):
                continue
            f_upper = f.upper()
            if prefix.upper() in f_upper and (f_upper.endswith(".TXT") or "." not in f):
                matched_files.append(os.path.join(root, f).replace("\\", "/"))
    return sorted(matched_files)

demo_files = collect_faers_files("DEMO")
drug_files = collect_faers_files("DRUG")
reac_files = collect_faers_files("REAC")

print(f"Discovered Files Across All Quarters (including 2014):")
print(f"  • DEMO files: {len(demo_files)}")
print(f"  • DRUG files: {len(drug_files)}")
print(f"  • REAC files: {len(reac_files)}\n")

def run_sglt2_pipeline():
    os.makedirs(DB_PATH.parent, exist_ok=True)
    con = duckdb.connect(str(DB_PATH))
    
    print("--- 1. Reading & Unifying DEMO Files (with union_by_name) ---")
    con.execute(f"""
        CREATE OR REPLACE TABLE raw_demo AS 
        SELECT * FROM read_csv_auto(
            {demo_files}, 
            delim='$', 
            header=True, 
            ignore_errors=True, 
            all_varchar=True,
            union_by_name=True
        );
    """)
    print(f"✓ DEMO loaded: {con.execute('SELECT COUNT(*) FROM raw_demo;').fetchone()[0]:,} total rows")

    print("\n--- 2. Reading & Filtering DRUG Files for SGLT2 Inhibitors ---")
    con.execute(f"""
        CREATE OR REPLACE TABLE raw_drug_sglt2 AS 
        SELECT 
            primaryid,
            caseid,
            UPPER(drugname) AS drugname,
            UPPER(role_cod) AS role_cod
        FROM read_csv_auto(
            {drug_files}, 
            delim='$', 
            header=True, 
            ignore_errors=True, 
            all_varchar=True,
            union_by_name=True
        )
        WHERE UPPER(role_cod) = 'PS'
          AND (
               UPPER(drugname) LIKE '%JARDIANCE%' 
            OR UPPER(drugname) LIKE '%EMPAGLIFLOZIN%'
            OR UPPER(drugname) LIKE '%FARXIGA%' 
            OR UPPER(drugname) LIKE '%DAPAGLIFLOZIN%'
            OR UPPER(drugname) LIKE '%INVOKANA%' 
            OR UPPER(drugname) LIKE '%CANAGLIFLOZIN%'
            OR UPPER(drugname) LIKE '%STEGLATRO%'
            OR UPPER(drugname) LIKE '%ERTUGLIFLOZIN%'
          );
    """)
    print(f"✓ Filtered DRUG loaded: {con.execute('SELECT COUNT(*) FROM raw_drug_sglt2;').fetchone()[0]:,} SGLT2 records")

    print("\n--- 3. Reading REAC Files ---")
    con.execute(f"""
        CREATE OR REPLACE TABLE raw_reac AS 
        SELECT 
            primaryid,
            caseid,
            UPPER(pt) AS adverse_event
        FROM read_csv_auto(
            {reac_files}, 
            delim='$', 
            header=True, 
            ignore_errors=True, 
            all_varchar=True,
            union_by_name=True
        );
    """)
    print(f"✓ REAC loaded: {con.execute('SELECT COUNT(*) FROM raw_reac;').fetchone()[0]:,} rows")

    print("\n--- 4. Deduplicating DEMO (Latest FDA_DT / Highest PrimaryID) ---")
    con.execute("""
        CREATE OR REPLACE TABLE clean_demo AS
        WITH ranked AS (
            SELECT *,
                   ROW_NUMBER() OVER (
                       PARTITION BY caseid 
                       ORDER BY TRY_CAST(fda_dt AS INTEGER) DESC NULLS LAST, 
                                TRY_CAST(primaryid AS BIGINT) DESC
                   ) as rank
            FROM raw_demo
        )
        SELECT * EXCLUDE (rank) FROM ranked WHERE rank = 1;
    """)
    print(f"✓ Deduplicated DEMO: {con.execute('SELECT COUNT(*) FROM clean_demo;').fetchone()[0]:,} unique cases")

    print("\n--- 5. Creating SGLT2 Cohort Table ---")
    con.execute("""
        CREATE OR REPLACE TABLE sglt2_cohort AS
        SELECT 
            d.primaryid,
            d.caseid,
            TRY_CAST(d.age AS DOUBLE) AS age,
            d.age_cod,
            d.sex,
            d.reporter_country,
            d.fda_dt,
            dr.drugname,
            dr.role_cod,
            r.adverse_event
        FROM raw_drug_sglt2 dr
        JOIN clean_demo d ON dr.primaryid = d.primaryid
        JOIN raw_reac r ON dr.primaryid = r.primaryid;
    """)
    
    total_sglt2_cases = con.execute("SELECT COUNT(DISTINCT caseid) FROM sglt2_cohort;").fetchone()[0]
    total_sglt2_pairs = con.execute("SELECT COUNT(*) FROM sglt2_cohort;").fetchone()[0]
    
    print(f"\n🎉 SGLT2 Ingestion Complete!")
    print(f"  • SGLT2 Unique Cases: {total_sglt2_cases:,}")
    print(f"  • SGLT2 Case-Event Rows: {total_sglt2_pairs:,}")
    
    con.close()

run_sglt2_pipeline()

Discovered Files Across All Quarters (including 2014):
  • DEMO files: 24
  • DRUG files: 24
  • REAC files: 24

--- 1. Reading & Unifying DEMO Files (with union_by_name) ---
✓ DEMO loaded: 8,309,659 total rows

--- 2. Reading & Filtering DRUG Files for SGLT2 Inhibitors ---
✓ Filtered DRUG loaded: 40,067 SGLT2 records

--- 3. Reading REAC Files ---
✓ REAC loaded: 25,084,108 rows

--- 4. Deduplicating DEMO (Latest FDA_DT / Highest PrimaryID) ---
✓ Deduplicated DEMO: 7,443,175 unique cases

--- 5. Creating SGLT2 Cohort Table ---

🎉 SGLT2 Ingestion Complete!
  • SGLT2 Unique Cases: 36,983
  • SGLT2 Case-Event Rows: 84,756


## GLP-1 Receptor Agonists

In [26]:
## GLP-1 Receptor Agonists
def collect_faers_files(prefix):
    matched_files = []
    for root, _, files in os.walk(RAW_DATA_DIR):
        for f in files:
            if f.startswith('.'):
                continue
            f_upper = f.upper()
            if prefix.upper() in f_upper and (f_upper.endswith(".TXT") or "." not in f):
                matched_files.append(os.path.join(root, f).replace("\\", "/"))
    return sorted(matched_files)

demo_files = collect_faers_files("DEMO")
drug_files = collect_faers_files("DRUG")
reac_files = collect_faers_files("REAC")

print(f"Discovered Files Across All Quarters :")
print(f"  • DEMO files: {len(demo_files)}")
print(f"  • DRUG files: {len(drug_files)}")
print(f"  • REAC files: {len(reac_files)}\n")

def run_glp1_pipeline():
    os.makedirs(DB_PATH.parent, exist_ok=True)
    con = duckdb.connect(str(DB_PATH))
    
    print("--- 1. Reading & Unifying DEMO Files (with union_by_name) ---")
    con.execute(f"""
        CREATE OR REPLACE TABLE raw_demo AS 
        SELECT * FROM read_csv_auto(
            {demo_files}, 
            delim='$', 
            header=True, 
            ignore_errors=True, 
            all_varchar=True,
            union_by_name=True
        );
    """)
    print(f"✓ DEMO loaded: {con.execute('SELECT COUNT(*) FROM raw_demo;').fetchone()[0]:,} total rows")

    print("\n--- 2. Reading & Filtering DRUG Files for GLP1 Inhibitors ---")
    con.execute(f"""
        CREATE OR REPLACE TABLE raw_drug_glp1 AS 
        SELECT 
            primaryid,
            caseid,
            UPPER(drugname) AS drugname,
            UPPER(role_cod) AS role_cod
        FROM read_csv_auto(
            {drug_files}, 
            delim='$', 
            header=True, 
            ignore_errors=True, 
            all_varchar=True,
            union_by_name=True
        )
        WHERE UPPER(role_cod) = 'PS'
          AND (
               UPPER(drugname) LIKE '%SAXENDA%' 
            OR UPPER(drugname) LIKE '%VICTOZA%'
            OR UPPER(drugname) LIKE '%LIRAGLUTIDE%'
            OR UPPER(drugname) LIKE '%MOUNJARO%' 
            OR UPPER(drugname) LIKE '%TIRZEPATIDE%'
            OR UPPER(drugname) LIKE '%RYBELSUS%' 
            OR UPPER(drugname) LIKE '%WEGOVY%'
            OR UPPER(drugname) LIKE '%OZEMPIC%'
            OR UPPER(drugname) LIKE '%SEMAGLUTIDE%'
            OR UPPER(drugname) LIKE '%TRULICITY%'
            OR UPPER(drugname) LIKE '%DULAGLUTIDE%'
          );
    """)
    print(f"✓ Filtered DRUG loaded: {con.execute('SELECT COUNT(*) FROM raw_drug_glp1;').fetchone()[0]:,} GLP1 records")

    print("\n--- 3. Reading REAC Files ---")
    con.execute(f"""
        CREATE OR REPLACE TABLE raw_reac AS 
        SELECT 
            primaryid,
            caseid,
            UPPER(pt) AS adverse_event
        FROM read_csv_auto(
            {reac_files}, 
            delim='$', 
            header=True, 
            ignore_errors=True, 
            all_varchar=True,
            union_by_name=True
        );
    """)
    print(f"✓ REAC loaded: {con.execute('SELECT COUNT(*) FROM raw_reac;').fetchone()[0]:,} rows")

    print("\n--- 4. Deduplicating DEMO (Latest FDA_DT / Highest PrimaryID) ---")
    con.execute("""
        CREATE OR REPLACE TABLE clean_demo AS
        WITH ranked AS (
            SELECT *,
                   ROW_NUMBER() OVER (
                       PARTITION BY caseid 
                       ORDER BY TRY_CAST(fda_dt AS INTEGER) DESC NULLS LAST, 
                                TRY_CAST(primaryid AS BIGINT) DESC
                   ) as rank
            FROM raw_demo
        )
        SELECT * EXCLUDE (rank) FROM ranked WHERE rank = 1;
    """)
    print(f"✓ Deduplicated DEMO: {con.execute('SELECT COUNT(*) FROM clean_demo;').fetchone()[0]:,} unique cases")

    print("\n--- 5. Creating GLP1 Cohort Table ---")
    con.execute("""
        CREATE OR REPLACE TABLE glp1_cohort AS
        SELECT 
            d.primaryid,
            d.caseid,
            TRY_CAST(d.age AS DOUBLE) AS age,
            d.age_cod,
            d.sex,
            d.reporter_country,
            d.fda_dt,
            dr.drugname,
            dr.role_cod,
            r.adverse_event
        FROM raw_drug_glp1 dr
        JOIN clean_demo d ON dr.primaryid = d.primaryid
        JOIN raw_reac r ON dr.primaryid = r.primaryid;
    """)
    
    total_glp1_cases = con.execute("SELECT COUNT(DISTINCT caseid) FROM glp1_cohort;").fetchone()[0]
    total_glp1_pairs = con.execute("SELECT COUNT(*) FROM glp1_cohort;").fetchone()[0]
    
    print(f"\n🎉 GLP1 Ingestion Complete!")
    print(f"  • GLP1 Unique Cases: {total_glp1_cases:,}")
    print(f"  • GLP1 Case-Event Rows: {total_glp1_pairs:,}")
    
    con.close()

run_glp1_pipeline()

Discovered Files Across All Quarters :
  • DEMO files: 24
  • DRUG files: 24
  • REAC files: 24

--- 1. Reading & Unifying DEMO Files (with union_by_name) ---
✓ DEMO loaded: 8,309,659 total rows

--- 2. Reading & Filtering DRUG Files for GLP1 Inhibitors ---
✓ Filtered DRUG loaded: 42,509 GLP1 records

--- 3. Reading REAC Files ---
✓ REAC loaded: 25,084,108 rows

--- 4. Deduplicating DEMO (Latest FDA_DT / Highest PrimaryID) ---
✓ Deduplicated DEMO: 7,443,175 unique cases

--- 5. Creating GLP1 Cohort Table ---

🎉 GLP1 Ingestion Complete!
  • GLP1 Unique Cases: 41,266
  • GLP1 Case-Event Rows: 96,217


## Deduplication & Creating Database with required info
Deduplication & save both sglt2_case_events and glp1_case_events (along with their unique case views) directly into the faers_db.duckdb file.

In [ ]:


def build_cohort_database():
    os.makedirs(DB_PATH.parent, exist_ok=True)
    con = duckdb.connect(str(DB_PATH))
    
    print("--- 1. Loading and Deduplicating Raw DEMO Table ---")
    con.execute(f"""
        CREATE OR REPLACE TABLE raw_demo AS 
        SELECT * FROM read_csv_auto(
            {demo_files}, 
            delim='$', header=True, ignore_errors=True, all_varchar=True, union_by_name=True
        );
    """)
    
    # Deduplicate DEMO: Pick latest fda_dt, then highest primaryid for each caseid
    con.execute("""
        CREATE OR REPLACE TABLE clean_demo AS
        WITH ranked AS (
            SELECT *,
                   ROW_NUMBER() OVER (
                       PARTITION BY caseid 
                       ORDER BY TRY_CAST(fda_dt AS INTEGER) DESC NULLS LAST, 
                                TRY_CAST(primaryid AS BIGINT) DESC
                   ) as rank
            FROM raw_demo
        )
        SELECT * EXCLUDE (rank) FROM ranked WHERE rank = 1;
    """)
    print(f"✓ Deduplicated DEMO: {con.execute('SELECT COUNT(*) FROM clean_demo;').fetchone()[0]:,} unique cases")

    print("\n--- 2. Staging Filtered DRUG & REAC Tables ---")
    # Filter SGLT2 Drugs (Primary Suspect 'PS')
    con.execute(f"""
        CREATE OR REPLACE TABLE raw_drug_sglt2 AS 
        SELECT primaryid, caseid, UPPER(drugname) AS drugname, UPPER(role_cod) AS role_cod
        FROM read_csv_auto({drug_files}, delim='$', header=True, ignore_errors=True, all_varchar=True, union_by_name=True)
        WHERE UPPER(role_cod) = 'PS'
          AND (
               UPPER(drugname) LIKE '%JARDIANCE%' OR UPPER(drugname) LIKE '%EMPAGLIFLOZIN%'
            OR UPPER(drugname) LIKE '%FARXIGA%' OR UPPER(drugname) LIKE '%DAPAGLIFLOZIN%'
            OR UPPER(drugname) LIKE '%INVOKANA%' OR UPPER(drugname) LIKE '%CANAGLIFLOZIN%'
            OR UPPER(drugname) LIKE '%STEGLATRO%' OR UPPER(drugname) LIKE '%ERTUGLIFLOZIN%'
          );
    """)

    # Filter GLP-1 Drugs (Primary Suspect 'PS')
    con.execute(f"""
        CREATE OR REPLACE TABLE raw_drug_glp1 AS 
        SELECT primaryid, caseid, UPPER(drugname) AS drugname, UPPER(role_cod) AS role_cod
        FROM read_csv_auto({drug_files}, delim='$', header=True, ignore_errors=True, all_varchar=True, union_by_name=True)
        WHERE UPPER(role_cod) = 'PS'
          AND (
               UPPER(drugname) LIKE '%OZEMPIC%' OR UPPER(drugname) LIKE '%SEMAGLUTIDE%'
            OR UPPER(drugname) LIKE '%WEGOVY%' OR UPPER(drugname) LIKE '%RYBELSUS%'
            OR UPPER(drugname) LIKE '%MOUNJARO%' OR UPPER(drugname) LIKE '%TIRZEPATIDE%'
            OR UPPER(drugname) LIKE '%VICTOZA%' OR UPPER(drugname) LIKE '%LIRAGLUTIDE%'
            OR UPPER(drugname) LIKE '%TRULICITY%' OR UPPER(drugname) LIKE '%DULAGLUTIDE%'
            OR UPPER(drugname) LIKE '%SAXENDA%' 
          );
    """)

    # Stage REAC Table
    con.execute(f"""
        CREATE OR REPLACE TABLE raw_reac AS 
        SELECT primaryid, caseid, UPPER(pt) AS adverse_event
        FROM read_csv_auto({reac_files}, delim='$', header=True, ignore_errors=True, all_varchar=True, union_by_name=True);
    """)

    print("\n--- 3. Creating SGLT2 Cohort Tables & Views ---")
    con.execute("""
        CREATE OR REPLACE TABLE sglt2_case_events AS
        SELECT 
            d.caseid,
            d.primaryid,
            TRY_CAST(d.age AS DOUBLE) AS age,
            d.age_cod,
            d.sex,
            d.reporter_country,
            d.fda_dt,
            dr.drugname,
            dr.role_cod,
            r.adverse_event
        FROM raw_drug_sglt2 dr
        JOIN clean_demo d ON dr.primaryid = d.primaryid
        JOIN raw_reac r ON dr.primaryid = r.primaryid;
        
        CREATE OR REPLACE VIEW sglt2_unique_cases AS
        SELECT DISTINCT caseid, primaryid, age, age_cod, sex, reporter_country, fda_dt, drugname
        FROM sglt2_case_events;
    """)
    print(f"✓ SGLT2 Case-Event Rows: {con.execute('SELECT COUNT(*) FROM sglt2_case_events;').fetchone()[0]:,}")
    print(f"✓ SGLT2 Unique Cases: {con.execute('SELECT COUNT(*) FROM sglt2_unique_cases;').fetchone()[0]:,}")

    print("\n--- 4. Creating GLP-1 Cohort Tables & Views ---")
    con.execute("""
        CREATE OR REPLACE TABLE glp1_case_events AS
        SELECT 
            d.caseid,
            d.primaryid,
            TRY_CAST(d.age AS DOUBLE) AS age,
            d.age_cod,
            d.sex,
            d.reporter_country,
            d.fda_dt,
            dr.drugname,
            dr.role_cod,
            r.adverse_event
        FROM raw_drug_glp1 dr
        JOIN clean_demo d ON dr.primaryid = d.primaryid
        JOIN raw_reac r ON dr.primaryid = r.primaryid;
        
        CREATE OR REPLACE VIEW glp1_unique_cases AS
        SELECT DISTINCT caseid, primaryid, age, age_cod, sex, reporter_country, fda_dt, drugname
        FROM glp1_case_events;
    """)
    print(f"✓ GLP-1 Case-Event Rows: {con.execute('SELECT COUNT(*) FROM glp1_case_events;').fetchone()[0]:,}")
    print(f"✓ GLP-1 Unique Cases: {con.execute('SELECT COUNT(*) FROM glp1_unique_cases;').fetchone()[0]:,}")

    # Drop temporary raw staging tables to save storage space
    con.execute("DROP TABLE IF EXISTS raw_demo;")
    con.execute("DROP TABLE IF EXISTS raw_drug_sglt2;")
    con.execute("DROP TABLE IF EXISTS raw_drug_glp1;")
    
    con.close()
    print("\n🎉 Database pipeline finished! Tables and views persist in `faers_db.duckdb`.")

build_cohort_database()

--- 1. Loading and Deduplicating Raw DEMO Table ---
✓ Deduplicated DEMO: 7,443,175 unique cases

--- 2. Staging Filtered DRUG & REAC Tables ---

--- 3. Creating SGLT2 Cohort Tables & Views ---
✓ SGLT2 Case-Event Rows: 84,756
✓ SGLT2 Unique Cases: 36,983

--- 4. Creating GLP-1 Cohort Tables & Views ---
✓ GLP-1 Case-Event Rows: 96,217
✓ GLP-1 Unique Cases: 41,266

🎉 Database pipeline finished! Tables and views persist in `faers_db.duckdb`.


In [ ]:
# verify the database state in Python

con = duckdb.connect(str(DB_PATH))

print("--- Database Tables & Views ---")
display(con.execute("SHOW TABLES;").df())

print("\n--- Summary Row Counts ---")
print(f"SGLT2 Case-Events:  {con.execute('SELECT COUNT(*) FROM sglt2_case_events;').fetchone()[0]:,}")
print(f"SGLT2 Unique Cases: {con.execute('SELECT COUNT(*) FROM sglt2_unique_cases;').fetchone()[0]:,}")
print(f"GLP-1 Case-Events:  {con.execute('SELECT COUNT(*) FROM glp1_case_events;').fetchone()[0]:,}")
print(f"GLP-1 Unique Cases: {con.execute('SELECT COUNT(*) FROM glp1_unique_cases;').fetchone()[0]:,}")

con.close()

--- Database Tables & Views ---


,name
0,clean_demo
1,glp1_case_events
2,glp1_cohort
3,glp1_unique_cases
4,raw_drug
5,raw_reac
6,sglt2_case_events
7,sglt2_cohort
8,sglt2_unique_cases



--- Summary Row Counts ---
SGLT2 Case-Events:  84,756
SGLT2 Unique Cases: 36,983
GLP-1 Case-Events:  96,217
GLP-1 Unique Cases: 41,266


In [2]:
import duckdb

# Explicitly close any ambient DuckDB connections
try:
    duckdb.close()
    print("✓ Closed ambient DuckDB connection.")
except Exception as e:
    print(f"Notice: {e}")

✓ Closed ambient DuckDB connection.
